# 02 · Custom dataloader for fine-tuning
Uses the manifest from notebook 01 to build a `TableJsonDataset`, a padding `Collator`, and train/val `DataLoader`s. We inspect one batch's tensor shapes.

**Sample layout:** `input_ids = [prompt tokens] + [target JSON tokens]`; `labels` mask the prompt with `-100` so the loss is computed only on the JSON.

In [ ]:
# --- Bootstrap: make the package importable without installing, and stay OFFLINE.
import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


In [ ]:
from gemma_ft_json.config import load_config
from gemma_ft_json.tokenization import build_tokenizer
from gemma_ft_json.data.transforms import build_image_transform
from gemma_ft_json.data.dataset import TableJsonDataset, IGNORE_INDEX
from gemma_ft_json.data.collate import build_dataloaders

cfg = load_config(ROOT / 'configs' / 'default.yaml')
man = ROOT / 'data' / 'demo' / 'manifests'
train_path, val_path = man / 'train.jsonl', man / 'val.jsonl'
assert train_path.is_file(), 'Run notebook 01 first to create the manifests.'
# Keep the demo light + offline.
cfg.model.vision.image_size = 128; cfg.model.vision.patch_size = 16
cfg.data.max_target_tokens = 128

In [ ]:
# Tokenizer is chosen by the decoder backend: 'stub' -> ByteTokenizer (offline),
# 'local_gemma' -> the local Gemma tokenizer (offline). Both expose the same API.
tokenizer = build_tokenizer(cfg.model.decoder)
transform = build_image_transform(cfg.model.vision)
print('tokenizer:', type(tokenizer).__name__, '| vocab_size:', tokenizer.vocab_size,
      '| pad/bos/eos:', tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id)

In [ ]:
train_ds = TableJsonDataset(train_path, tokenizer, transform,
                            prompt=cfg.data.prompt, max_target_tokens=cfg.data.max_target_tokens)
val_ds   = TableJsonDataset(val_path, tokenizer, transform,
                            prompt=cfg.data.prompt, max_target_tokens=cfg.data.max_target_tokens)
print('train/val sizes:', len(train_ds), len(val_ds))
sample = train_ds[0]
print({k: tuple(v.shape) for k, v in sample.items()})

In [ ]:
train_loader, val_loader = build_dataloaders(
    train_ds, val_ds, pad_id=tokenizer.pad_id,
    batch_size=cfg.dataloader.batch_size, num_workers=0,
    pin_memory=False, shuffle_train=cfg.dataloader.shuffle_train,
)
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f'{k:15s} {tuple(v.shape)}  {v.dtype}')

### Decode the supervised (non `-100`) part of a sample back to text

In [ ]:
ids = batch['input_ids'][0].tolist()
labels = batch['labels'][0].tolist()
supervised = [i for i, l in zip(ids, labels) if l != IGNORE_INDEX]
print('TARGET the model must produce:')
print(tokenizer.decode(supervised))

These exact loaders are what notebook **04** trains on.